# Milestone 8 - Designing an eval: which prompt should ship?

Milestone 7 looked inside the model. This one is about the question you'll face
every week when you build with LLMs: **I changed something. Is it actually
better?**

You'll run one complete, honest experiment, start to finish:

1. **Write the plan** before seeing any result.
2. **Look at the eval set**: its groups, and how big a difference it can detect.
3. **Run both setups** on the same examples.
4. **Score by group**, with a range for each score.
5. **Compare case by case** (a paired comparison, McNemar's rule).
6. **Bootstrap** a range for the difference.
7. **Decide**, using the rule from step 1.

**The task.** A support app sorts incoming messages into `billing`, `shipping`,
`refund` or `fraud`, so they reach the right team. Prompt **A** is the one in use:
a single line. Prompt **B** adds a short definition of each category. B is
several times longer, so every message costs more to sort. Is B worth it?

Each idea here also has a Daily Drill lesson (in the *Experiment design* and
*Evals* modules), and `docs/EXPERIMENT_PLAN.md` is the template this notebook
follows.

**Do you need Ollama?** It's best with it, running `gemma3:12b`. Without it,
the notebook uses answers recorded from that model on a MacBook Pro, so it still
works; you just can't change the prompts.

In [ ]:
import json
import os
import random
import re
import urllib.request

import ipytest
import pytest

ipytest.autoconfig(clean="test_*")

OLLAMA = "http://localhost:11434"
MODEL = "gemma3:12b"


def ollama_is_running():
    if os.getenv("DRILL_NOTEBOOK_OFFLINE") == "1":   # the test suite sets this
        return False
    try:
        urllib.request.urlopen(OLLAMA + "/api/version", timeout=1)
        return True
    except OSError:
        return False


LIVE = ollama_is_running()
print("Live: talking to Ollama on this computer." if LIVE
      else "Offline: using recorded answers. Start Ollama and re-run this cell to go live.")


def ask(prompt, model=MODEL):
    """One chat message to Ollama at temperature 0, returning the reply text."""
    body = {"model": model, "stream": False, "think": False,
            "options": {"temperature": 0, "num_predict": 8},
            "messages": [{"role": "user", "content": prompt}]}
    request = urllib.request.Request(OLLAMA + "/api/chat", json.dumps(body).encode(),
                                     {"content-type": "application/json"})
    return json.load(urllib.request.urlopen(request, timeout=300))["message"]["content"]


# --- Helpers from the Daily Drill lessons, ready to use ---------------------------

def wilson(passed, total, z=1.96):
    """95% range for a pass rate ("Margins near 0% and 100%")."""
    if total == 0:
        return (0.0, 1.0)
    p = passed / total
    denominator = 1 + z * z / total
    centre = (p + z * z / (2 * total)) / denominator
    half = z * ((p * (1 - p) / total + z * z / (4 * total * total)) ** 0.5) / denominator
    return (max(0.0, centre - half), min(1.0, centre + half))


def mcnemar_verdict(a_only, b_only):
    """"Comparing on the same examples": a winner only if |gap| >= 2 * sqrt(disagreements)."""
    disagreements = a_only + b_only
    gap = a_only - b_only
    if disagreements == 0 or gap * gap < 4 * disagreements:
        return "too close to call"
    return "A" if gap > 0 else "B"


def weighted(rates, shares):
    """"Report each group, weighted by real traffic"."""
    total = sum(shares[g] for g in rates)
    return sum(rates[g] * shares[g] for g in rates) / total

In [ ]:
# The eval set: 96 labelled support messages. The last four in each group are
# deliberately hard, sitting between two categories.
MESSAGES = {
 "billing": [
  "Why did my monthly plan go up from $12 to $15?",
  "Can I switch my payment card to a different Visa?",
  "I need a copy of my invoice for March for my expense report.",
  "Is there a discount if I pay yearly instead of monthly?",
  "My payment failed but I have money in the account. What happened?",
  "How do I add my company's tax number to my invoices?",
  "When will I be charged for the next billing period?",
  "Can you explain the 'service fee' line on my bill?",
  "I want to downgrade to the basic plan starting next month.",
  "Do you accept PayPal for the subscription?",
  "The invoice shows the wrong company name, can you fix it?",
  "I was charged in US dollars but I live in Canada, can I pay in CAD?",
  "How do I update the billing address on my account?",
  "What happens if my card expires before the renewal date?",
  "Can I get receipts emailed to my accountant as well?",
  "Is the price shown including sales tax?",
  "I'd like to pay by bank transfer instead of card.",
  "Why is there a pending charge of $1 on my card?",
  "Can I split the payment across two cards?",
  "My trial ended, how much will the first month cost?",
  "Please send me a statement of all charges this year.",
  "How do I cancel auto-renew without cancelling right now?",
  "The checkout page keeps saying my postal code is invalid.",
  "Do students get a lower price?",
  "Can I move my plan to my partner's account and pay from there?",
  "I see two plans on my account, which one am I paying for?",
  "I was charged twice for the same subscription this month.",
  "You billed me after I cancelled, please stop charging me.",
  "My card was declined but you still sent a payment reminder.",
  "Why do I have a charge from you that says \"adjustment\"?"
 ],
 "shipping": [
  "Where is my order? It was supposed to arrive Tuesday.",
  "The tracking number you sent doesn't work on the courier's site.",
  "Can I change the delivery address? I'm moving next week.",
  "My package says delivered but it's not at my door.",
  "Do you ship to the Yukon?",
  "How long does standard shipping take to Vancouver?",
  "The box arrived crushed, the contents seem fine though.",
  "Can I pick up my order instead of having it delivered?",
  "Why has my parcel been stuck in Toronto for four days?",
  "Is express shipping available for this item?",
  "The courier left a note saying they'll try again tomorrow, can I choose a time?",
  "I only received one of the two items I ordered, is the other coming separately?",
  "Can you deliver to my office instead of my home?",
  "My order hasn't shipped yet and it's been a week.",
  "Do I need to be home to sign for the delivery?",
  "The tracking hasn't updated since Monday.",
  "Can you hold my package at the post office?",
  "I typed my apartment number wrong on the order.",
  "Which courier do you use for rural addresses?",
  "Is there a way to combine my two orders into one shipment?",
  "The delivery estimate changed from Friday to next Wednesday, why?",
  "Do you deliver on weekends?",
  "My neighbour got my package by mistake, what should I do?",
  "Can I add delivery instructions to leave it at the back door?",
  "My order came to the wrong address and the person there won't give it back.",
  "The courier says the package was returned to you, what now?",
  "I paid for express but it took nine days.",
  "Can I cancel the delivery and pick it up myself? It's stuck at the depot."
 ],
 "refund": [
  "The jacket doesn't fit, how do I return it for my money back?",
  "I'd like a refund for the headphones, they stopped working after a week.",
  "I returned the item two weeks ago and still haven't got my money.",
  "Can I get a refund instead of store credit?",
  "The product is not as described, I want to send it back.",
  "How long do refunds take to show on my card?",
  "I cancelled my order before it shipped, when will I be refunded?",
  "Do I have to pay for return shipping?",
  "Part of my order was missing, can you refund that item?",
  "I bought the wrong size, can I return it even though I opened it?",
  "My refund was less than what I paid, why?",
  "Is there a deadline for returning items?",
  "The item arrived broken, I don't want a replacement, just my money back.",
  "Can I return a gift without the receipt?",
  "I was refunded to an old card I no longer have, what happens now?",
  "How do I print a return label?",
  "I want to return the blender, it's too loud.",
  "Can I get a partial refund if I keep the damaged chair?",
  "You charged me for an item you said was out of stock, please refund it.",
  "I changed my mind about the order, can I still get my money back?",
  "I was charged twice for one order, please give me one of the payments back.",
  "My package never arrived, I just want my money back.",
  "You billed me for shipping on an order that was supposed to ship free, refund the difference please.",
  "The subscription box I cancelled still came, I'll send it back for my money."
 ],
 "fraud": [
  "There are three orders on my account that I didn't place.",
  "I got an email asking me to confirm my password on a site that looks like yours, is it real?",
  "Someone changed the email on my account and I can't log in.",
  "My card was charged by your company but I've never shopped with you.",
  "I received a text saying I won a prize from you and need to pay a fee to claim it.",
  "A package was sent to an address I don't recognise, using my account.",
  "Someone called pretending to be your support and asked for my card number.",
  "My saved card was used for a $900 purchase I didn't make.",
  "I got a password reset email I didn't request, three times today.",
  "There's a new device logged into my account from another country.",
  "I didn't order this, but a package with my name just arrived and now there's a charge.",
  "My account shows a refund request I never made.",
  "Someone in my household used my card on your site without asking.",
  "I got an invoice from you for a plan I never signed up for, and the email isn't mine."
 ]
}

# Each group's share of real traffic. Fraud is rare, so the eval set has extra.
TRAFFIC = {"billing": 0.45, "shipping": 0.35, "refund": 0.15, "fraud": 0.05}

PROMPT_A = "Which category is this customer message: billing, shipping, refund or fraud? Reply with one word.\n\n{message}"

PROMPT_B = "Sort this customer message into exactly one category.\n\nbilling: charges, prices, invoices, payment methods, plans\nshipping: delivery, tracking, addresses, couriers, delays\nrefund: returns and getting money back for an order\nfraud: activity the customer did not do, scams, phishing, account takeover\n\nReply with the category name only.\n\nMessage: {message}"

# gemma3:12b's answers to every message with each prompt, recorded on a MacBook Pro.
RECORDED = {"A": {"billing-01": "billing", "billing-02": "billing", "billing-03": "billing", "billing-04": "billing", "billing-05": "billing", "billing-06": "billing", "billing-07": "billing", "billing-08": "billing", "billing-09": "billing", "billing-10": "billing", "billing-11": "billing", "billing-12": "billing", "billing-13": "billing", "billing-14": "billing", "billing-15": "billing", "billing-16": "billing", "billing-17": "billing", "billing-18": "billing", "billing-19": "billing", "billing-20": "billing", "billing-21": "billing", "billing-22": "billing", "billing-23": "billing", "billing-24": "billing", "billing-25": "billing", "billing-26": "billing", "billing-27": "billing", "billing-28": "billing", "billing-29": "billing", "billing-30": "billing", "shipping-01": "shipping", "shipping-02": "shipping", "shipping-03": "shipping", "shipping-04": "shipping", "shipping-05": "shipping", "shipping-06": "shipping", "shipping-07": "shipping", "shipping-08": "shipping", "shipping-09": "shipping", "shipping-10": "shipping", "shipping-11": "shipping", "shipping-12": "shipping", "shipping-13": "shipping", "shipping-14": "shipping", "shipping-15": "shipping", "shipping-16": "shipping", "shipping-17": "shipping", "shipping-18": "shipping", "shipping-19": "shipping", "shipping-20": "shipping", "shipping-21": "shipping", "shipping-22": "shipping", "shipping-23": "shipping", "shipping-24": "shipping", "shipping-25": "shipping", "shipping-26": "shipping", "shipping-27": "billing", "shipping-28": "shipping", "refund-01": "refund", "refund-02": "refund", "refund-03": "refund", "refund-04": "refund", "refund-05": "refund", "refund-06": "refund", "refund-07": "refund", "refund-08": "billing", "refund-09": "refund", "refund-10": "refund", "refund-11": "refund", "refund-12": "shipping", "refund-13": "refund", "refund-14": "refund", "refund-15": "refund", "refund-16": "shipping", "refund-17": "refund", "refund-18": "refund", "refund-19": "refund", "refund-20": "refund", "refund-21": "billing", "refund-22": "refund", "refund-23": "billing", "refund-24": "refund", "fraud-01": "fraud", "fraud-02": "fraud", "fraud-03": "fraud", "fraud-04": "fraud", "fraud-05": "fraud", "fraud-06": "fraud", "fraud-07": "fraud", "fraud-08": "fraud", "fraud-09": "fraud", "fraud-10": "fraud", "fraud-11": "fraud", "fraud-12": "fraud", "fraud-13": "fraud", "fraud-14": "billing"}, "B": {"billing-01": "billing", "billing-02": "billing", "billing-03": "billing", "billing-04": "billing", "billing-05": "billing", "billing-06": "billing", "billing-07": "billing", "billing-08": "billing", "billing-09": "billing", "billing-10": "billing", "billing-11": "billing", "billing-12": "billing", "billing-13": "billing", "billing-14": "billing", "billing-15": "billing", "billing-16": "billing", "billing-17": "billing", "billing-18": "billing", "billing-19": "billing", "billing-20": "billing", "billing-21": "billing", "billing-22": "billing", "billing-23": "shipping", "billing-24": "billing", "billing-25": "billing", "billing-26": "billing", "billing-27": "billing", "billing-28": "billing", "billing-29": "billing", "billing-30": "billing", "shipping-01": "shipping", "shipping-02": "shipping", "shipping-03": "shipping", "shipping-04": "shipping", "shipping-05": "shipping", "shipping-06": "shipping", "shipping-07": "shipping", "shipping-08": "shipping", "shipping-09": "shipping", "shipping-10": "shipping", "shipping-11": "shipping", "shipping-12": "shipping", "shipping-13": "shipping", "shipping-14": "shipping", "shipping-15": "shipping", "shipping-16": "shipping", "shipping-17": "shipping", "shipping-18": "shipping", "shipping-19": "shipping", "shipping-20": "shipping", "shipping-21": "shipping", "shipping-22": "shipping", "shipping-23": "shipping", "shipping-24": "shipping", "shipping-25": "shipping", "shipping-26": "shipping", "shipping-27": "shipping", "shipping-28": "shipping", "refund-01": "refund", "refund-02": "refund", "refund-03": "refund", "refund-04": "refund", "refund-05": "refund", "refund-06": "refund", "refund-07": "refund", "refund-08": "refund", "refund-09": "refund", "refund-10": "refund", "refund-11": "refund", "refund-12": "refund", "refund-13": "refund", "refund-14": "refund", "refund-15": "refund", "refund-16": "refund", "refund-17": "refund", "refund-18": "refund", "refund-19": "refund", "refund-20": "refund", "refund-21": "billing", "refund-22": "refund", "refund-23": "refund", "refund-24": "refund", "fraud-01": "fraud", "fraud-02": "fraud", "fraud-03": "fraud", "fraud-04": "fraud", "fraud-05": "fraud", "fraud-06": "fraud", "fraud-07": "fraud", "fraud-08": "fraud", "fraud-09": "fraud", "fraud-10": "fraud", "fraud-11": "fraud", "fraud-12": "fraud", "fraud-13": "fraud", "fraud-14": "billing"}}

## 1. Write the plan first

Before running anything, decide how you'll judge the result. If you pick the
rule after seeing the numbers, you'll pick the rule that favours what you saw,
usually without noticing. Writing it first is called **pre-registration**.

Read this plan carefully. Every later step follows it, and the final step
applies its decision rule mechanically.

In [ ]:
PLAN = {
    "question": "Does adding category definitions (prompt B) sort messages more accurately than prompt A?",
    "change": "the prompt; model, temperature (0) and messages stay the same",
    "eval_set": "milestone 8 support messages, 96 labelled, 4 groups",
    "metric": "share sorted correctly, per group and weighted by real traffic",
    "runs": "1 per message: at temperature 0 the model gives the same answer every time",
    "test": "paired, McNemar's rule",
    "decision_rule": "Switch to B only if it wins clearly by McNemar's rule. "
                     "B costs more per message, so 'too close to call' means keep A.",
    "no_result": "Add more hard refund and fraud messages, and run again.",
}
for field, value in PLAN.items():
    print(f"{field:14} {value}")

**Why one run per message?** Repeat runs measure randomness. At temperature 0
there is none: the same prompt gets the same answer. If you sort at a higher
temperature in production, evaluate at that temperature and run each message
several times.

## 2. Look at the eval set before using it

**Question.** How are the 96 messages spread across the groups, compared with
real traffic? And how big a difference can 96 messages detect?

**Your guess.** The quick rule from the Daily Drill: a pass rate on n examples
is good to about ±100 ÷ √n points. Work it out for 96 before running the cell.

In [ ]:
total = sum(len(msgs) for msgs in MESSAGES.values())
print(f"{'group':10} {'messages':>9} {'share of eval set':>18} {'share of traffic':>17} {'margin, about':>14}")
for group, msgs in MESSAGES.items():
    print(f"{group:10} {len(msgs):9} {len(msgs) / total:18.0%} {TRAFFIC[group]:17.0%} "
          f"{'±' + str(round(100 / len(msgs) ** 0.5)) + ' points':>14}")
print(f"{'all':10} {total:9} {'':18} {'':17} {'±' + str(round(100 / total ** 0.5)) + ' points':>14}")

**What happened.** Fraud is 5% of traffic but about 15% of the eval set: it's
rare but costly to get wrong, so it gets extra examples on purpose. That's
**stratifying**, and it's why the headline score will be weighted by traffic
later, not by these counts.

The margins are sobering. Overall, 96 messages give about ±10 points. Per
group it's worse, ±27 points for fraud's 14 messages. So this eval set can show
a big difference but not a small one, and that's worth knowing *before* you
run it.

In a real project you would also split the set into **dev** (to iterate on),
**test** (to check before shipping) and a locked **holdout**, as in the Daily
Drill lesson "Dev, test and holdout sets". This notebook keeps one set so the
numbers are big enough to discuss.

## 3. Run both prompts on the same messages

Same model, same temperature, same messages; only the prompt differs. The
model's reply is cut down to the first category word it contains, and anything
else counts as wrong.

**Your guess.** Out of 96, how many will each prompt get right?

In [ ]:
CATEGORIES = list(MESSAGES)


def first_category(reply):
    """The first category word in a reply, or "unknown"."""
    for word in re.findall(r"[a-z]+", reply.lower()):
        if word in CATEGORIES:
            return word
    return "unknown"


def sort_all(prompt, name):
    if not LIVE:
        return RECORDED[name]
    return {f"{group}-{i + 1:02d}": first_category(ask(prompt.format(message=message)))
            for group, msgs in MESSAGES.items() for i, message in enumerate(msgs)}


ANSWERS = {"A": sort_all(PROMPT_A, "A"), "B": sort_all(PROMPT_B, "B")}
IDS = list(ANSWERS["A"])                      # the same messages, in the same order, for both
TRUE = {i: i.split("-")[0] for i in IDS}      # the id starts with the right group
PASSED = {name: [ANSWERS[name][i] == TRUE[i] for i in IDS] for name in ANSWERS}

for name in ANSWERS:
    print(f"prompt {name}: {sum(PASSED[name])} of {len(IDS)} correct")

**What happened.** On the recorded run, prompt A got 89 right and B got 93.
B *looks* four points better. Hold that thought: the margin from step 2 was
about ±10 points.

## 4. Score each group, with a range

One overall number can hide a group in trouble. Score each group separately,
and give each score a range. The `wilson` helper from the setup cell gives a
95% range that behaves well near 100%, which is where these scores are.

**Your turn.** Write `group_counts(results)`. `results` is a list of
`(group, passed)` pairs. Return a dict from each group to `[passed, total]`.

**How to start.** It's the aggregation pattern from milestone 4: a dict of
running totals. For each pair, make sure the group has an entry, add one to its
total, and add one to its passed count if it passed.

In [ ]:
def group_counts(results):
    """Return {group: [passed, total]} from a list of (group, passed) pairs."""
    # TODO: one dict entry per group, holding [passed, total]; update it for each pair.
    raise NotImplementedError("Milestone 8: implement group_counts")

In [ ]:
%%ipytest -qq

def test_counts_per_group():
    results = [("billing", True), ("fraud", False), ("billing", False), ("fraud", False)]
    assert group_counts(results) == {"billing": [1, 2], "fraud": [0, 2]}


def test_empty_results():
    assert group_counts([]) == {}


def test_all_passed():
    assert group_counts([("refund", True)] * 3) == {"refund": [3, 3]}

In [ ]:
BY_GROUP = {name: group_counts([(TRUE[i], ok) for i, ok in zip(IDS, PASSED[name])]) for name in ANSWERS}

print(f"{'group':10} {'A':>8} {'95% range':>13}   {'B':>8} {'95% range':>13}")
for group in CATEGORIES:
    row = f"{group:10}"
    for name in ANSWERS:
        passed, n = BY_GROUP[name][group]
        low, high = wilson(passed, n)
        row += f" {passed:3}/{n:<4} {low:5.0%}-{high:4.0%}  "
    print(row)

for name in ANSWERS:
    rates = {g: p / n for g, (p, n) in BY_GROUP[name].items()}
    plain = sum(PASSED[name]) / len(IDS)
    print(f"prompt {name}: plain average {plain:.1%}, weighted by traffic {weighted(rates, TRAFFIC):.1%}")

**What happened.** Almost all of the difference is in **refunds**: on the
recorded run, B sorted 23 of 24 refund messages correctly and A only 19. Most
of A's misses were about **returns**: "Do I have to pay for return shipping?"
and "How do I print a return label?" went to shipping or billing. B's
definition, "refund: returns and getting money back for an order", fixes exactly
that. Both prompts still miss "I was charged twice for one order, please give
me one of the payments back", which really does sit between billing and refund.
Reading the failures like this is how you find what to fix next.

The ranges are wide, though, because each group is small. And the weighted
score moves less than the plain average, because refunds are only 15% of real
traffic.

## 5. Compare case by case

Both prompts sorted the same 96 messages, so compare them **message by
message**. Where both are right, or both wrong, a message says nothing about
which prompt is better. Only the disagreements count.

**Your guess.** Of the messages where A and B disagree, how many did each get
right?

**Your turn.** Write `paired_counts(a_passed, b_passed)`: two lists of booleans
for the same messages in the same order. Return `[a_only, b_only]`: how many
only A got right, and how many only B got right.

**How to start.** `zip(a_passed, b_passed)` walks both lists together. Count
the two kinds of disagreement and skip the rest.

In [ ]:
def paired_counts(a_passed, b_passed):
    """Return [a_only, b_only]: messages only A got right, and only B got right."""
    # TODO: walk both lists together; count where exactly one of them passed.
    raise NotImplementedError("Milestone 8: implement paired_counts")

In [ ]:
%%ipytest -qq

def test_only_disagreements_count():
    a = [True, True, False, False, True]
    b = [True, False, True, False, False]
    assert paired_counts(a, b) == [2, 1]


def test_identical_results_have_no_disagreements():
    assert paired_counts([True, False], [True, False]) == [0, 0]


def test_empty():
    assert paired_counts([], []) == [0, 0]

In [ ]:
a_only, b_only = paired_counts(PASSED["A"], PASSED["B"])
print(f"only A right: {a_only}   only B right: {b_only}   disagreements: {a_only + b_only}")
print(f"McNemar's rule: the gap must be at least 2 x sqrt({a_only + b_only}) = {2 * (a_only + b_only) ** 0.5:.1f}; "
      f"it is {abs(a_only - b_only)}")
print("verdict:", mcnemar_verdict(a_only, b_only))

**What happened.** On the recorded run, B was right on 5 messages that A got
wrong, and A on 1 that B got wrong. 5 to 1 *feels* convincing. But with only 6
disagreements, McNemar's rule needs a gap of about 4.9, and the gap is 4. **Too
close to call.**

That's not "B is no better". It's "this eval set can't tell yet." Six
disagreements is very little evidence, and a 5-to-1 split happens by luck more
often than you'd think.

### Read the regressions and improvements

Counting the disagreements decided the comparison. **Reading** them explains it.
An **improvement** is a message B got right and A got wrong; a **regression** is
the reverse. (Statisticians call both together **discordant pairs**.) Here they
are, with both answers:

In [ ]:
CHANGED = [i for i in IDS if PASSED["A"][IDS.index(i)] != PASSED["B"][IDS.index(i)]]
for i in CHANGED:
    group, n = i.split("-")
    kind = "improvement" if ANSWERS["B"][i] == TRUE[i] else "regression"
    print(f"{kind:12} {i:12} A: {ANSWERS['A'][i]:9} B: {ANSWERS['B'][i]:9} {MESSAGES[group][int(n) - 1]}")

For each one, ask: **what in prompt B made the difference?** Then write the cause
in a few words. Here are the causes for the recorded run; if you're running
live and see one without a label, add your own.

In [ ]:
CAUSES = {
    "refund-08": "'returns' in B's refund definition",
    "refund-12": "'returns' in B's refund definition",
    "refund-16": "'returns' in B's refund definition",
    "refund-23": "'money back for an order' in B's refund definition",
    "shipping-27": "'delays' in B's shipping definition",
    "billing-23": "'addresses' in B's shipping definition caught a checkout problem",
}

for kind in ["improvement", "regression"]:
    counts = {}
    for i in CHANGED:
        if (ANSWERS["B"][i] == TRUE[i]) == (kind == "improvement"):
            cause = CAUSES.get(i, "unlabelled")
            counts[cause] = counts.get(cause, 0) + 1
    print(kind + "s:", counts)

**What happened.** Most improvements come from one word: B's refund definition
mentions **returns**, and A kept sending return questions to shipping or billing.
The one regression is just as useful. B's shipping definition mentions
**addresses**, which pulled "my postal code is invalid at checkout" away from
billing.

That's what reading them gives you that a score can't: the next change to try.
A prompt C that adds "checkout" to the billing definition should keep the
improvements and remove the regression. Testing that is the next experiment.
This method is called **error analysis**, and the Daily Drill lesson "Reading
regressions and improvements" practises it.

## 6. Bootstrap a range

The **bootstrap** gives a range for any score, with no formula: rebuild the
eval set many times by drawing messages at random, **with repeats allowed**, and
score each rebuilt set. The middle 95% of those scores is your range.

Do it on the *difference* per message (B right minus A right: +1, 0 or -1) and
you get a range for how much better B is. That's a **paired bootstrap**. If
the range includes 0, "no difference" is still possible.

**Your turn.** Write `bootstrap_range(values, draws, seed)`:

1. Make a random generator: `rng = random.Random(seed)`. The seed makes it
   repeatable.
2. Repeat `draws` times: build a sample of `len(values)` items, each one
   `values[rng.randrange(len(values))]`, and keep its average.
3. Sort the averages, drop the lowest 2.5% and the highest 2.5%: with
   `cut = int(draws * 0.025)`, the range is `[averages[cut], averages[draws - 1 - cut]]`.

Round both ends to 3 decimal places. For an empty list, return `[0.0, 0.0]`.

In [ ]:
def bootstrap_range(values, draws, seed):
    """Middle 95% of the averages of `draws` resamples of values, as [low, high]."""
    # TODO: resample with repeats `draws` times, keep each average, sort, and cut 2.5% off each end.
    raise NotImplementedError("Milestone 8: implement bootstrap_range")

In [ ]:
%%ipytest -qq

def test_same_seed_same_range():
    values = [1] * 30 + [0] * 10
    assert bootstrap_range(values, 1000, seed=7) == bootstrap_range(values, 1000, seed=7)


def test_range_surrounds_the_average():
    low, high = bootstrap_range([1] * 80 + [0] * 20, 1000, seed=1)
    assert 0.65 < low < 0.8 < high < 0.92


def test_more_data_gives_a_narrower_range():
    small = bootstrap_range([1] * 8 + [0] * 2, 1000, seed=1)
    large = bootstrap_range([1] * 80 + [0] * 20, 1000, seed=1)
    assert large[1] - large[0] < small[1] - small[0]


def test_no_variation_means_no_range():
    assert bootstrap_range([1, 1, 1], 200, seed=3) == [1.0, 1.0]


def test_empty():
    assert bootstrap_range([], 100, seed=1) == [0.0, 0.0]

In [ ]:
for name in ANSWERS:
    low, high = bootstrap_range([1 if ok else 0 for ok in PASSED[name]], 2000, seed=42)
    print(f"prompt {name}: {sum(PASSED[name]) / len(IDS):.1%} correct, bootstrap range {low:.1%} to {high:.1%}")

differences = [int(b) - int(a) for a, b in zip(PASSED["A"], PASSED["B"])]
low, high = bootstrap_range(differences, 2000, seed=42)
print(f"B minus A: {sum(differences) / len(IDS):+.1%}, paired bootstrap range {low:+.1%} to {high:+.1%}")

**What happened.** On the recorded run, B is about 4 points ahead, and the
paired range for the difference runs from 0 to about 9 points. Its low end sits
right at zero, which tells the same story as McNemar's rule: probably a real
improvement, not yet proven. Two different methods agreeing is a good sign
you've read the evidence correctly.

## 7. Decide, using the rule you wrote first

Now apply the plan's decision rule, exactly as written, whatever you'd *like*
the answer to be.

In [ ]:
verdict = mcnemar_verdict(a_only, b_only)
if verdict == "B":
    decision = "Switch to prompt B."
elif verdict == "A":
    decision = "Keep prompt A: it is clearly better."
else:
    decision = "Keep prompt A for now. " + PLAN["no_result"]
print("decision rule:", PLAN["decision_rule"])
print("verdict:      ", verdict)
print("decision:     ", decision)

**What happened.** On the recorded run: keep A, and add more hard refund and
fraud messages. That can feel unsatisfying, because B is *probably* better. But
the plan said that B, being more expensive, has to win clearly, and it hasn't
yet.

The analysis also showed you **where to collect evidence next**: refunds. Add,
say, 30 more refund messages that sit near billing, run again, and B will
either win clearly or turn out to be no better. Either way, the decision you
make then will be one you can defend.

**What a real team would add:**

- **Log it.** Save the plan, the counts and the verdict with the run in MLflow,
  so anyone can see the rule came before the result:

  ```python
  with mlflow.start_run(run_name="sort-prompt-a-vs-b"):
      mlflow.log_dict(PLAN, "plan.json")
      mlflow.log_metrics({"a_only": a_only, "b_only": b_only})
      mlflow.set_tag("verdict", verdict)
  ```

- **Check the labels.** Here the right answers were written by one person. In
  practice, have two people label a sample independently and check their
  agreement with Cohen's kappa before trusting the labels.
- **Keep a holdout.** Before shipping a change, confirm it once on examples you
  never looked at while choosing.

**What to take away from milestone 8.**

| Step | The habit |
|---|---|
| Plan | Write the metric and the decision rule before running anything |
| Eval set | Know its groups and its margin before using it |
| Per group | Report every group with a range; weight the headline by real traffic |
| Paired | Compare on the same examples; only disagreements count |
| Bootstrap | Get a range for any number, including a difference |
| Decide | Apply the rule as written, and use the analysis to plan the next round |